In [72]:
print('ritu')

ritu


this is part of the langgraph work flow

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    feedback: str
    topic: str
    action: Literal[ "continue_slide", "update_outline", "update_slide", "complete", '' ]
    tool_caller: Literal["generate_outline","generate_slide_detail"]
class OutlineSlide(BaseModel):
    slide_number: int
    slide_title: str
class OutlineOutput(BaseModel):
    title: str
    total_slides: int
    slides: List[OutlineSlide]
class DetailedPoint(BaseModel):
    key_point: str
    explanation: str
class DetailedSlideOutput(BaseModel):
    slide_number: int
    slide_title: str
    layout: Literal[
        "bullets",
        "bullets_with_text",
        "paragraph",
        "two_column",
        "mixed"
    ]
    intro_line: Optional[str] = None
    bullet_points: Optional[List[str]] = None
    supporting_text: Optional[str] = None
    paragraphs: Optional[List[str]] = None

model = ChatGroq(model="llama-3.3-70b-versatile")

searchTool = TavilySearchResults(max_results=3)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)
outline_parser = PydanticOutputParser(pydantic_object= OutlineOutput)
# outline_parser = PydanticOutputParser(pydantic_object= OutlineSlide)
detailed_parser = PydanticOutputParser(pydantic_object= DetailedSlideOutput)
OUTLINE_SYSTEM_PROMPT = SystemMessage(
    content=f"""You are an expert presentation designer.

Your task:
Generate ONLY the presentation title and slide titles.
    
{outline_parser.get_format_instructions()} 

Strict Rules:
- Generate EXACTLY the number of slides requested by the user.
- Do NOT generate key points.
- Do NOT generate slide content.
- Only generate slide_number and slide_title.
- Slide numbers must start from 1 and increment sequentially.
- Ensure logical flow from introduction to conclusion.
- Keep slide titles concise but descriptive.
- Use tools only if factual accuracy is required.
- Return ONLY valid JSON.
- No markdown.
- No explanations.
""")


DETAIL_SYSTEM_PROMPT = SystemMessage(
content=f"""
You are a professional presentation designer.

Your job is to generate visually balanced slide content for a PowerPoint presentation.

For each slide choose the most appropriate layout or a mix of layouts  and generate structured content.

Available layouts:
- bullets
- bullets_with_text
- paragraph
- mixed


{detailed_parser.get_format_instructions()}

General Principles:
Slides should be informative but not crowded.
A good slide often combines short explanations with bullet points.

Content Elements:
Slides may contain combination of:
- intro_line (a introductory sentence)
- bullet_points (key ideas)
- supporting_text (a short insight or explanation)
- paragraphs (short explanation text)


Layout Guidelines:

bullets
- 6-8 bullet points
- each bullet 6–12 words
- may optionally include a intro_line before bullets
- Some bullet slides should include a short explanation sentence before or after the bullets.

bullets_with_text
- 5-8 bullet points
- include supporting_text (1 short explanation sentence)
- may optionally include intro_line
- Some bullet slides should include a short explanation sentence before or after the bullets.

paragraph
- 3-5 short paragraphs
- each paragraph max 40 words
- optionally include a short intro_line

Slides may contain a combination of:

- intro_line (1 explanation sentence)
- bullet_points (3–5 bullets)
- paragraphs (1 short paragraph)
- supporting_text (1 insight sentence)

Good slides often combine elements.
Example structure:

intro_line
bullet_points
supporting_text


Layout Distribution Rules:

- 40–50% slides → bullets
- 20–30% slides → bullets_with_text
- 10–20% slides → paragraph

Variation Rules:
Do NOT generate the same layout repeatedly.
Use different content styles across slides.


Content Density Rules:
Slides should contain roughly 80-120 words total.
Content must fit comfortably on a PowerPoint slide.

Quality Rules:
- Bullet points should be concise and informative.
- Avoid repeating the same wording.
- Ensure the slide content is clear when presented visually.
Return JSON only.
"""
)


def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    messages = state["messages"] + [OUTLINE_SYSTEM_PROMPT]
    result = model_with_tools.invoke(messages)
    output = {
        'messages':[result],
        'current_slide_index':0,
        "tool_caller": "generate_outline",
            } 
    if result.content:
        try:
            output['outline'] = outline_parser.parse(repair_json(str(result.content))).model_dump()
            print("output['outline']",output['outline'])
        except json.JSONDecodeError as e:
            print('generate_outline_node',e)
    return output
def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    detailed_slides = state.get('detailed_slides',[])
    total_slides = len(state.get('outline',{}).get('slides',[]))
    current_slide = outline['slides'][int(current_index)]
    if current_index >= total_slides or current_slide['slide_title'] is None:
        return {"action": "complete"}
    output = {
        "tool_caller": "generate_slide_detail",
        "current_slide_index":current_index
    }
    if state['action'] == "update_slide":
        feedback = state['feedback']
        last_slide = detailed_slides.pop()
        last_outline = outline['slides'][current_index-1]
        output['feedback'] = ''
        output['action'] = ''
        prompt = HumanMessage(
            content=f"""
Update this slide content.

Presentation Title:
{outline['title']}

Outline of the slide:
{last_outline}

Current Slide Content:
{last_slide}

User Feedback:
{feedback}
"""
        )

    else:
        
        
        prompt = HumanMessage(
            content=f"""Generate detailede content for this slide:
Presentation Title: {outline['title']}
Slide Title: {current_slide['slide_title']}
Slide Number: {current_slide['slide_number']}

Provide comprehensive, presentation-ready content."""
    )
    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    if isinstance( state['messages'][-1],ToolMessage):
        messages = state['messages'][-2:]+[DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    if result.content:
        try:
            detailed_slides.append(detailed_parser.parse(repair_json(str(result.content))).model_dump())
            output['detailed_slides'] = detailed_slides 
            output['current_slide_index'] = current_index +1
            
        except json.JSONDecodeError as e:
            print('generate_slide_detail_node inside',e)
    if output['current_slide_index'] == total_slides:
        output["action"] =  "complete"
    output['messages'] = [result]
    return output


this  is the data that is gererate on topic Plastic Pollution and Its Effect on the Environment where number of slide is variable lenght

[
    {
        'slide_number': 1,
        'slide_title': 'Introduction to Plastic Pollution',
        'layout': 'mixed',
        'intro_line': 'Plastic pollution has become a major environmental concern worldwide.',
        'bullet_points': [
            'Causes harm to marine life',
            'Contaminates the food chain',
            'Affects human health'
        ],
        'supporting_text': "Plastic waste is a significant threat to our ecosystem, and it's essential to address this issue to ensure a sustainable future.",
    },
    {
        'slide_number': 2,
        'slide_title': 'Causes of Plastic Pollution',
        'layout': 'bullets_with_text',
        'intro_line': 'Plastic pollution is a complex issue with multiple causes.',
        'bullet_points': [
            'Improper waste disposal',
            'Single-use plastics',
            'Lack of recycling infrastructure',
            'Plastic production exceeding demand',
            'Insufficient education and awareness'
        ],
        'supporting_text': 'These factors contribute to the overwhelming amount of plastic waste in our environment, highlighting the need for a multi-faceted approach to address the problem.',
    },
    {
        'slide_number': 3,
        'slide_title': 'Effects on Marine Life',
        'layout': 'bullets_with_text',
        'intro_line': 'Plastic pollution has severe consequences for marine life, from entanglement to ingestion.',
        'bullet_points': [
            'Entanglement in plastic debris',
            'Ingestion of microplastics',
            'Habitat destruction due to plastic waste'
        ],
        'supporting_text': 'These effects can lead to injury, suffocation, and even death, highlighting the urgent need for plastic pollution reduction.',
    },
    {
        'slide_number': 4,
        'slide_title': 'Human Health Impacts',
        'layout': 'mixed',
        'intro_line': 'Plastic pollution poses significant risks to human health, from chemical exposure to the spread of diseases.',
        'bullet_points': [
            'Chemical toxicity from plastics and additives',
            'Microplastics ingestion and potential physical harm',
            'Increased risk of certain cancers and reproductive issues'
        ],
        'supporting_text': 'The impact of plastic pollution on human health is a growing concern, with studies indicating that exposure to toxic chemicals from plastics can lead to a range of health problems, including cancer and reproductive issues. Furthermore, the ingestion of microplastics has been shown to potentially cause physical harm and inflammation in the body.',
        'paragraphs': [
            'Plastic pollution is a global health crisis that requires immediate attention and action. The production, use, and disposal of plastics release toxic chemicals into the environment, which can then be ingested or inhaled by humans, causing a range of health problems.',
            'The effects of plastic pollution on human health are far-reaching and can have devastating consequences, from birth defects and developmental problems to cancer and premature death.'
        ]
    },
    {
        'slide_number': 5,
        'slide_title': 'Solutions and Conclusion',
        'layout': 'mixed',
        'intro_line': 'To mitigate the effects of plastic pollution, a multi-faceted approach is necessary, involving individuals, organizations, and governments.',
        'bullet_points': [
            'Reduce plastic use and waste',
            'Increase recycling and proper waste disposal',
            'Implement extended producer responsibility',
            'Promote education and awareness'
        ],
        'supporting_text': 'By working together and implementing these solutions, we can significantly reduce plastic pollution and create a healthier environment for future generations.',
    }
]



this is a well structure json data but problem is that work or model is not generaring real time data, it not covering importend insight, no statistics, no repost, no reserach

topic can be any thing it not only to the Plastic Pollution it can be 
how to learn python
photosynthetic

Importance of Mental Health in Modern Life
Urbanization and Its Impact on Cities
Healthy Lifestyle and the Role of Exercise
Electric Vehicles and the Future of Transportation
The Importance of Time Management for Students
Global Warming: Causes and Solutions
Women Empowerment in Modern Society
The Role of Education in Personal Development

for every topic there must be a any reserach or ony organization that relest importent insite and outcome of research and statistics 

model should also include this valuale inside 


I will give you some code and prompt you need to first understand them and dont generate any intermideat output untill i ask you any quesion